# 🌿 Plant Disease Detector — Model Retraining v5

ShuffleNetV2 x1.0 · PlantVillage pretrain → PlantDoc fine-tune · honest train/val/test split · TFLite export

**Setup:** Settings → Accelerator → **GPU T4 x1** → Save.

Each code cell below opens with a plain-English banner explaining *what it does and why*.
The flow is: **configure → prep the photos → build the model → pretrain → train → grade → package for the app.**

## Step 0 — Configuration & reproducibility

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 0 · CONFIGURATION
# Every tunable number lives here so you never have to hunt through the notebook
# to change one. We also seed all the random-number generators BEFORE anything
# random happens — that makes the train/val split and the image augmentations
# come out identical on every run, so a change in accuracy reflects a real
# change you made, not luck of the draw.
# ═══════════════════════════════════════════════════════════════════════════
import os, random
import numpy as np
import torch

# ---- reproducibility ---------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# (Full bit-for-bit determinism would also need deterministic cuDNN kernels, which
#  are much slower. Seeding the data/augmentation RNG is enough to make runs comparable.)

# ---- where the data lives ----------------------------------------------------
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)
PLANTDOC_TRAIN = 'PlantDoc-Dataset/train'
PLANTDOC_TEST  = 'PlantDoc-Dataset/test'

# Optional PlantVillage pretraining. Attach the dataset (right panel → Add Input)
# and point this at the folder holding one subfolder per class. If the path doesn't
# exist, Step 6 quietly skips pretraining instead of crashing.
USE_PLANTVILLAGE_PRETRAIN = True
PLANTVILLAGE_DIR = '/kaggle/input/plantvillage-dataset/color'  # ← set to YOUR attached path

# ---- hyperparameters (the training "dials") ----------------------------------
IMG_SIZE       = 224       # all images are resized/cropped to 224×224
BATCH_SIZE     = 32        # images processed together per step
VAL_FRACTION   = 0.20      # 20% of the TRAIN folder is held out as a validation set
FREEZE_EPOCHS  = 10        # epochs of "head-only" warmup before unfreezing the backbone
NUM_EPOCHS     = 60        # max total epochs on PlantDoc (early stopping usually ends sooner)
PATIENCE       = 10        # stop if validation accuracy doesn't improve for this many epochs
PV_EPOCHS      = 5         # PlantVillage pretraining epochs (only if enabled)
LR_HEAD        = 1e-3      # learning rate during warmup
LR_FULL        = 3e-4      # learning rate once the whole network trains
WEIGHT_DECAY   = 1e-2      # regularization — discourages over-confident weights
NUM_WORKERS    = 2         # parallel data-loading processes

print('✅ Config loaded. Seed =', SEED)

## Step 1 — Check GPU

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 1 · GPU CHECK
# Training does millions of matrix multiplies; a GPU does them ~10-50× faster than
# a CPU. This just confirms Kaggle actually handed us one and picks the device
# every later cell will run on. If it prints ❌, the accelerator isn't enabled.
# ═══════════════════════════════════════════════════════════════════════════
if torch.cuda.is_available():
    print(f'✅ GPU ready: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU — go to Settings → Accelerator → GPU T4 x1 → Save')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Step 2 — Install dependencies

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 2 · INSTALL THE TFLITE CONVERTER
# Everything else (PyTorch, torchvision) is preinstalled on Kaggle. The one extra
# tool is ai-edge-torch — Google's official PyTorch→TFLite converter. We use it
# instead of the older PyTorch→ONNX→TFLite route, which was buggy and sometimes
# produced a model with corrupted weights.
# ═══════════════════════════════════════════════════════════════════════════
!pip install ai-edge-torch -q
print('✅ Dependencies installed')

## Step 3 — Download & clean PlantDoc

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 3 · GET THE REAL-WORLD DATASET (and clean it)
# Downloads PlantDoc — the in-the-wild leaf photos that match what the app will
# actually see. Two cleanup jobs happen here:
#   1. We only treat real FOLDERS as classes (a stray README or hidden file would
#      otherwise become a fake "class" and crash the counting loop).
#   2. PlantDoc ships one extra training class that has no test images (and, as it
#      turns out, only ~2 training images). You can't learn or grade such a class,
#      so we delete it. That also keeps train/test label numbering perfectly aligned
#      — otherwise ImageFolder would hand the same disease different label numbers
#      in train vs test and every test score would be wrong.
# ═══════════════════════════════════════════════════════════════════════════
if not os.path.exists('PlantDoc-Dataset'):
    !git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
else:
    print('PlantDoc already downloaded')

def list_class_dirs(root):
    # Return only the real subdirectories of `root`, sorted — never stray files.
    return sorted(d for d in os.listdir(root)
                  if os.path.isdir(os.path.join(root, d)))

def count_images(root, class_dirs):
    return sum(
        sum(1 for f in os.listdir(os.path.join(root, c))
            if os.path.isfile(os.path.join(root, c, f)))
        for c in class_dirs
    )

train_class_dirs = list_class_dirs(PLANTDOC_TRAIN)
test_class_dirs  = list_class_dirs(PLANTDOC_TEST)

# Drop any TRAIN class that has no matching TEST folder (can't be evaluated, and
# too few images to learn). Safe to re-run: once dropped, there's nothing left to do.
import shutil
for c in sorted(set(train_class_dirs) - set(test_class_dirs)):
    shutil.rmtree(os.path.join(PLANTDOC_TRAIN, c))
    print(f'🗑  Dropped train-only class: {c}')
train_class_dirs = list_class_dirs(PLANTDOC_TRAIN)

# After cleanup the two splits should describe the same set of classes.
assert set(train_class_dirs) == set(test_class_dirs), 'Train/test classes still differ!'

print('✅ Dataset ready')
print(f'   Classes : {len(train_class_dirs)}')
print(f'   Train   : {count_images(PLANTDOC_TRAIN, train_class_dirs)} images')
print(f'   Test    : {count_images(PLANTDOC_TEST,  test_class_dirs)} images')

## Step 4 — Image transforms, the train/val/test split, and data loaders

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 4 · PREPARE THE PHOTOS AND SPLIT THEM (the busiest setup cell)
# Three jobs:
#   A) Define how images are processed. TRAINING images get random crops, flips,
#      color shifts, blur, etc. — variety that stops the model from memorizing.
#      EVALUATION images get a plain, fixed resize+centre-crop so scoring is
#      consistent and geometrically comparable to the training crops.
#   B) Split the data into THREE sets:
#        • train      → what the model studies
#        • validation → a slice of the train folder used to pick the best epoch
#        • test       → PlantDoc's official held-out set, touched ONCE in Step 9
#      (The old version reused the test set for validation, which quietly inflated
#       the score. This split is the fix.)
#   C) Build DataLoaders. The train loader uses a WeightedRandomSampler so rare
#      diseases get shown as often as common ones, despite the class imbalance.
# ═══════════════════════════════════════════════════════════════════════════
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from sklearn.model_selection import train_test_split

# ImageNet normalization stats — required because the backbone was pretrained on ImageNet.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# --- A) transforms ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomRotation(20),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # blanks a random patch — robustness
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),       # matches the geometry of the training crop
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Two views over the SAME train folder so the train/val subsets can use different
# transforms (a Subset can't carry its own transform — it borrows its parent's).
train_full_aug  = datasets.ImageFolder(PLANTDOC_TRAIN, transform=train_transforms)
train_full_eval = datasets.ImageFolder(PLANTDOC_TRAIN, transform=eval_transforms)

# TRAIN defines the canonical class names and their integer labels.
CLASS_NAMES   = train_full_aug.classes
NUM_CLASSES   = len(CLASS_NAMES)
CANONICAL_C2I = train_full_aug.class_to_idx

class AlignedImageFolder(datasets.ImageFolder):
    # Force the test set to reuse TRAIN's class→index mapping. If a class were ever
    # missing here it would keep its canonical label and simply contribute no images,
    # instead of silently shifting every later label by one. (Belt-and-suspenders now
    # that Step 3 already aligned the folders.)
    def find_classes(self, directory):
        present = {d.name for d in os.scandir(directory) if d.is_dir()}
        classes = [c for c in CLASS_NAMES if c in present]
        return classes, {c: CANONICAL_C2I[c] for c in classes}

test_dataset = AlignedImageFolder(PLANTDOC_TEST, transform=eval_transforms)
for c, idx in test_dataset.class_to_idx.items():
    assert CANONICAL_C2I[c] == idx, f'label mismatch for {c}'

# --- B) stratified 80/20 split of the train folder (indices, not copies) ---
targets = [label for _, label in train_full_aug.samples]
train_idx, val_idx = train_test_split(
    np.arange(len(targets)), test_size=VAL_FRACTION,
    stratify=targets, random_state=SEED,   # stratify keeps each class's proportion
)
train_dataset = Subset(train_full_aug,  train_idx)   # augmented view
val_dataset   = Subset(train_full_eval, val_idx)     # plain eval view

# --- C) weighted sampler (computed over the TRAIN subset only) ---
train_labels = np.array(targets)[train_idx]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

print('Images per class (train subset):')
for i, (name, count) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  {i:2d}. {name:<45} {count}{"  ⚠️ EMPTY" if count == 0 else ""}')

# Inverse-frequency weights → rare classes are drawn more often. The clip guards
# against 1/0 = inf if a class ever ends up empty.
sample_weights = (1.0 / np.clip(class_counts, 1, None))[train_labels]
g = torch.Generator(); g.manual_seed(SEED)   # seeded so sampling is reproducible
sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double),
                                num_samples=len(train_idx), replacement=True, generator=g)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'\n✅ Data loaders ready')
print(f'   Train : {len(train_dataset)} images, {len(train_loader)} batches')
print(f'   Val   : {len(val_dataset)} images, {len(val_loader)} batches')
print(f'   Test  : {len(test_dataset)} images, {len(test_loader)} batches  (held out)')
print(f'   Classes: {NUM_CLASSES}')

## Step 5 — Build the model & shared train/eval helpers

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 5 · BUILD THE MODEL + REUSABLE HELPERS (nothing trains yet)
# • build_model: loads ShuffleNetV2 x1.0 (small & fast, ~1.3M params — chosen so the
#   exported model runs in real time on a phone) with ImageNet weights, then swaps
#   its final layer for Dropout + a Linear that outputs our NUM_CLASSES diseases.
# • freeze_backbone / unfreeze_all: toggle which layers learn (used for the two-phase
#   training in Step 7).
# • run_epoch: one full pass over a loader. Pass an optimizer → it trains; omit it →
#   it just evaluates (no gradients). Returns (loss, accuracy).
# ═══════════════════════════════════════════════════════════════════════════
import torch.nn as nn
from torchvision.models import shufflenet_v2_x1_0, ShuffleNet_V2_X1_0_Weights

def build_model(num_classes):
    m = shufflenet_v2_x1_0(weights=ShuffleNet_V2_X1_0_Weights.IMAGENET1K_V1)
    in_features = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Dropout(p=0.4),                       # randomly drops 40% of features → less overfitting
        nn.Linear(in_features, num_classes),     # the new disease classifier
    )
    return m

def freeze_backbone(m):
    # Only the new classifier head ('fc') learns; the borrowed feature extractor is frozen.
    for name, param in m.named_parameters():
        param.requires_grad = ('fc' in name)

def unfreeze_all(m):
    for param in m.parameters():
        param.requires_grad = True

def run_epoch(m, loader, criterion, optimizer=None):
    train_mode = optimizer is not None       # optimizer present → training; absent → evaluating
    m.train(train_mode)
    total_loss, correct, total = 0.0, 0, 0
    torch.set_grad_enabled(train_mode)       # no gradient bookkeeping during eval = faster
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if train_mode:
            optimizer.zero_grad()
        outputs = m(images)
        loss = criterion(outputs, labels)
        if train_mode:
            loss.backward()                  # compute gradients
            optimizer.step()                 # nudge the weights
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    torch.set_grad_enabled(True)
    return total_loss / total, correct / total * 100

criterion = nn.CrossEntropyLoss()            # standard loss for multi-class classification
print('✅ Model factory + helpers ready')

## Step 6 — (Optional) Pretrain the backbone on PlantVillage

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 6 · WARM UP ON THE BIG EASY DATASET (the accuracy trick)
# PlantDoc is small and hard. PlantVillage is large and easy (single leaves on clean
# backgrounds). So we first train a throwaway copy of the model on PlantVillage, then
# keep ONLY its feature-extracting backbone and discard its classifier head. Starting
# PlantDoc training from these "leaf-aware" features roughly doubles final accuracy
# vs. starting from plain ImageNet weights.
#
# IMPORTANT: PlantVillage is used for pretraining ONLY. Its clean lab images don't
# resemble real app photos, so we never validate or test on it. If the dataset isn't
# attached, this whole stage is skipped and we start from ImageNet weights instead.
# ═══════════════════════════════════════════════════════════════════════════
model = build_model(NUM_CLASSES).to(DEVICE)

pv_available = USE_PLANTVILLAGE_PRETRAIN and os.path.isdir(PLANTVILLAGE_DIR)

if pv_available:
    print(f'Pretraining on PlantVillage at {PLANTVILLAGE_DIR} ...')
    pv_dataset = datasets.ImageFolder(PLANTVILLAGE_DIR, transform=train_transforms)
    pv_loader  = DataLoader(pv_dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=True)

    # A temporary model with a head sized for PlantVillage's OWN class count.
    pv_model = build_model(len(pv_dataset.classes)).to(DEVICE)
    pv_opt = torch.optim.AdamW(pv_model.parameters(), lr=LR_FULL, weight_decay=WEIGHT_DECAY)
    for ep in range(PV_EPOCHS):
        tl, ta = run_epoch(pv_model, pv_loader, criterion, pv_opt)
        print(f'  PV epoch {ep+1}/{PV_EPOCHS}  train {ta:.1f}% (loss {tl:.3f})')

    # Copy everything EXCEPT the classifier head into our real model.
    pv_backbone = {k: v for k, v in pv_model.state_dict().items() if not k.startswith('fc.')}
    missing, _ = model.load_state_dict(pv_backbone, strict=False)
    print(f'✅ PlantVillage backbone transferred '
          f'(head left random; {len(missing)} head tensors untouched)')
    del pv_model, pv_loader, pv_dataset
    torch.cuda.empty_cache()
else:
    reason = 'disabled' if not USE_PLANTVILLAGE_PRETRAIN else f'not found at {PLANTVILLAGE_DIR}'
    print(f'⏭  Skipping PlantVillage pretraining ({reason}). Using ImageNet weights only.')

# Freeze the backbone so Step 7 starts with a head-only warmup.
freeze_backbone(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'   Total parameters : {total:,}')
print(f'   Trainable (head) : {trainable:,}')
print(f'   Output classes   : {NUM_CLASSES}')

## Step 7 — Train on PlantDoc (two phases + early stopping)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 7 · TRAIN ON PLANTDOC (the main event)
# Two phases:
#   • Phase 1 (epochs 1–FREEZE_EPOCHS): backbone frozen, only the new head learns,
#     at the higher LR_HEAD. A gentle warmup so the random head doesn't wreck the
#     borrowed features.
#   • Phase 2 (the rest): unfreeze everything and fine-tune the whole network at the
#     smaller LR_FULL, with a fresh cosine LR schedule over the remaining epochs.
# After every epoch we check accuracy on the VALIDATION set (never the test set),
# save the best-so-far model, and stop early if it hasn't improved for PATIENCE
# epochs. The number printed at the end is the best VALIDATION accuracy — the honest
# test number comes later in Step 9.
# ═══════════════════════════════════════════════════════════════════════════
import torch.optim as optim

# Phase-1 optimizer + LR schedule (head only).
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FREEZE_EPOCHS, eta_min=1e-6)

best_val_acc      = 0.0
epochs_no_improve = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):

    # ── Phase 1 → Phase 2 transition: unfreeze and restart the optimizer/schedule ──
    if epoch == FREEZE_EPOCHS:
        unfreeze_all(model)
        optimizer = optim.AdamW(model.parameters(), lr=LR_FULL, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=NUM_EPOCHS - FREEZE_EPOCHS, eta_min=1e-6)
        tn = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'\n🔓 Epoch {epoch+1}: backbone unfrozen — {tn:,} params now training, LR → {LR_FULL}\n')

    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)  # trains
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)             # evaluates
    scheduler.step()

    # Save the model only when validation accuracy hits a new high.
    if val_acc > best_val_acc:
        best_val_acc, epochs_no_improve = val_acc, 0
        torch.save(model.state_dict(), 'best_model.pth')
        saved = '  ✅ saved'
    else:
        epochs_no_improve += 1
        saved = f'  (no improve {epochs_no_improve}/{PATIENCE})'

    for k, v in zip(history, (train_loss, train_acc, val_loss, val_acc)):
        history[k].append(v)
    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS}  Train {train_acc:.1f}% (loss {train_loss:.3f})  '
          f'Val {val_acc:.1f}% (loss {val_loss:.3f}){saved}')

    if epochs_no_improve >= PATIENCE:        # early stopping
        print(f'\n⏹ Early stopping at epoch {epoch+1} — no val improvement for {PATIENCE} epochs')
        break

print(f'\n🏆 Best VALIDATION accuracy: {best_val_acc:.1f}%')

## Step 8 — Plot training curves

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 8 · PLOT THE LEARNING CURVES (a diagnostic, not a result)
# Two charts: accuracy and loss, each with a train line and a validation line.
# Read the GAP between them: a big gap (train high, val low) means the model is
# memorizing the small training set (overfitting) and more DATA would help more
# than more epochs; a small gap means it could train longer or go bigger.
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_acc']) + 1)

ax1.plot(epochs, history['train_acc'], 'b-o', label='Train', markersize=4)
ax1.plot(epochs, history['val_acc'],   'g-o', label='Validation', markersize=4)
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy (%)')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=4)
ax2.plot(epochs, history['val_loss'],   'g-o', label='Validation', markersize=4)
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

## Step 9 — Final evaluation on the held-out test set

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 9 · THE FINAL EXAM (the one honest number)
# Reloads the BEST saved checkpoint and runs it once over the held-out test set —
# images the model never saw during training or model selection. Prints overall
# accuracy plus a per-class precision/recall/F1 report, so you can see which
# diseases it handles well and which it confuses. This is the figure to trust.
# ═══════════════════════════════════════════════════════════════════════════
from sklearn.metrics import classification_report

model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.to(DEVICE).eval()

all_preds, all_labels = [], []
with torch.no_grad():                         # no training here — just predictions
    for images, labels in test_loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds, all_labels = np.array(all_preds), np.array(all_labels)
test_acc = (all_preds == all_labels).mean() * 100
print(f'🎯 Held-out TEST accuracy: {test_acc:.1f}%\n')

# zero_division=0 keeps the report clean for any class with no correct predictions.
print(classification_report(all_labels, all_preds,
                            labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, zero_division=0))

## Step 10 — Convert to TFLite

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 10 · EXPORT FOR THE PHONE
# Converts the trained PyTorch model into a single .tflite file your mobile app can
# run on-device. ai-edge-torch traces the model using one example input tensor to
# learn its shape, then writes the optimized graph. The model must be on the CPU for
# conversion.
# ═══════════════════════════════════════════════════════════════════════════
import ai_edge_torch

model.load_state_dict(torch.load('best_model.pth', map_location='cpu'))
model.eval().cpu()

sample_input = (torch.randn(1, 3, IMG_SIZE, IMG_SIZE),)   # one fake image to trace the shape

print('Converting... (may take 1-2 minutes)')
edge_model = ai_edge_torch.convert(model, sample_input)
edge_model.export('plant_disease_model.tflite')

size = os.path.getsize('plant_disease_model.tflite') / 1e6
print(f'✅ Conversion complete: plant_disease_model.tflite ({size:.1f} MB)')

## Step 11 — Generate classes.json

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 11 · WRITE THE LABEL LIST
# The model outputs a number (e.g. 14). This saves the class names in the EXACT
# order the model uses, so the app can turn "14" into a human label like
# "Soyabean leaf". Order matters — it must match the model's outputs exactly.
# ═══════════════════════════════════════════════════════════════════════════
import json

with open('classes.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)

print('✅ classes.json generated')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {i:2d}. {name}')

## Step 12 — Verify the TFLite model

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 12 · SANITY-CHECK THE EXPORTED MODEL
# Loads the .tflite file, feeds it a PROPERLY NORMALIZED dummy input (matching the
# eval pipeline, not raw 0-1 noise), and checks two things:
#   1. The outputs aren't inf/NaN (which would mean a broken conversion).
#   2. The TFLite outputs match the original PyTorch model on the same input.
# It also reports whether the exported input is "channels first" (NCHW) or
# "channels last" (NHWC) — your app's image preprocessing MUST feed tensors in this
# same layout, or predictions will be garbage.
# ═══════════════════════════════════════════════════════════════════════════
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path='plant_disease_model.tflite')
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('✅ TFLite model loaded')
print(f'   Input shape  : {input_details[0]["shape"]}')
print(f'   Input dtype  : {input_details[0]["dtype"]}')
print(f'   Output shape : {output_details[0]["shape"]}')

# A normalized sample in NCHW, then reshaped to whatever layout the export expects.
mean = np.array(IMAGENET_MEAN, dtype=np.float32).reshape(1, 3, 1, 1)
std  = np.array(IMAGENET_STD,  dtype=np.float32).reshape(1, 3, 1, 1)
nchw = ((np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)) - mean) / std

if list(input_details[0]['shape'])[1] == 3:      # NCHW export
    tflite_in, layout = nchw, 'NCHW (channels first)'
else:                                            # NHWC export
    tflite_in, layout = np.transpose(nchw, (0, 2, 3, 1)), 'NHWC (channels last)'

interpreter.set_tensor(input_details[0]['index'], tflite_in)
interpreter.invoke()
tflite_out = interpreter.get_tensor(output_details[0]['index'])

print('❌ TFLite outputs contain inf/NaN — conversion failed'
      if np.any(~np.isfinite(tflite_out)) else '✅ No inf/NaN in TFLite outputs')

with torch.no_grad():                            # compare against PyTorch on the same input
    torch_out = model(torch.from_numpy(nchw)).numpy()
max_diff = np.abs(torch_out - tflite_out).max()
print(f'   Max |PyTorch − TFLite| logit diff: {max_diff:.4e}  '
      f'({"OK" if max_diff < 1e-2 else "investigate"})')

print(f'\n⚠️  Exported input layout: {layout}')
print('   Make sure the app preprocessing feeds tensors in this layout.')

## Step 13 — Locate output files

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STEP 13 · FIND YOUR DELIVERABLES
# Everything was written straight into /kaggle/working, so there's nothing to copy.
# This just lists the files and reminds you where they go in the app.
# ═══════════════════════════════════════════════════════════════════════════
for fname in ['plant_disease_model.tflite', 'classes.json', 'training_curves.png']:
    path = os.path.join(WORK_DIR, fname)
    if os.path.exists(path):
        print(f'   {fname:<32} {os.path.getsize(path)/1e6:.2f} MB')

print('\nTo download:  Right panel → Output → download icon next to each file')
print('In the app:   assets/model/plant_disease_model.tflite')
print('              assets/model/classes.json')

## ✅ Done

- **Honest accuracy** = the Step 9 test number (the test set is used only once).
- **Step 7's number** is validation accuracy — it only drove early stopping / checkpointing.
- If accuracy looks low, that's expected for PlantDoc; the Step 8 curves show whether
  it's over- or under-fitting, and more/better real-world photos is the biggest lever.
- Double-check the input **layout** printed in Step 12 matches your app's preprocessing.